In [2]:
import numpy as np
import pandas as pd


# ============================================================
# Load window features
# ============================================================

window_features = pd.read_parquet(
    "../data/processed/window_features.parquet"
)


# ============================================================
# Metadata columns
# ============================================================

metadata_columns = [
    "identifier",
    "window",
    "window_start",
    "window_end",
    "phase",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
    "anomaly_label",
]


# ============================================================
# Model features
# ============================================================

model_features = [
    column
    for column in window_features.columns
    if column not in metadata_columns
]

print(f"Number of model features: {len(model_features)}")


# ============================================================
# Sort windows within each experiment
# ============================================================

window_features = (
    window_features
    .sort_values(["identifier", "window"])
    .reset_index(drop=True)
)


# ============================================================
# Previous-window features
# ============================================================

previous_features = (
    window_features
    .groupby("identifier", sort=False)[model_features]
    .shift(1)
)

previous_features.columns = [
    f"{column}_prev"
    for column in model_features
]


# ============================================================
# Delta features
# ============================================================

delta_values = (
    window_features[model_features].to_numpy()
    - previous_features.to_numpy()
)

delta_features = pd.DataFrame(
    delta_values,
    columns=[
        f"{column}_delta"
        for column in model_features
    ],
    index=window_features.index
)

# ============================================================
# Current + previous + delta features
# ============================================================

temporal_features = pd.concat(
    [
        window_features,
        previous_features,
        delta_features,
    ],
    axis=1
)


# ============================================================
# Validation
# ============================================================

print("\nTemporal feature matrix:")
print(temporal_features.shape)

print(
    f"Current features:  {len(model_features)}"
)
print(
    f"Previous features: {len(model_features)}"
)
print(
    f"Delta features:    {len(model_features)}"
)

print(
    f"\nExpected total columns: "
    f"{len(window_features.columns) + 2 * len(model_features)}"
)

print(
    f"Actual total columns:   "
    f"{len(temporal_features.columns)}"
)


# ============================================================
# Save to parquet
# ============================================================


temporal_features.to_parquet(
    "../data/processed/notebooks/temporal_features.parquet",
    index=False
)

print(f"Data saved to parquet!")


# ============================================================
# Check first window of each experiment
# ============================================================

first_windows = (
    temporal_features
    .groupby("identifier")
    .head(1)
)

previous_nan_count = (
    first_windows[
        [f"{column}_prev" for column in model_features]
    ]
    .isna()
    .all(axis=1)
    .sum()
)

delta_nan_count = (
    first_windows[
        [f"{column}_delta" for column in model_features]
    ]
    .isna()
    .all(axis=1)
    .sum()
)

print(
    f"\nExperiments with no previous window: "
    f"{previous_nan_count}"
)

print(
    f"Experiments with no delta for first window: "
    f"{delta_nan_count}"
)

Number of model features: 240

Temporal feature matrix:
(11512, 730)
Current features:  240
Previous features: 240
Delta features:    240

Expected total columns: 730
Actual total columns:   730
Data saved to parquet!

Experiments with no previous window: 119
Experiments with no delta for first window: 119
